In [1]:
import os
import re
import pandas as pd

In [2]:
RESULTS_DIR = "./results"
STATES_LIST = [3, 4, 5]

In [3]:
def extract_accuracy(file_path):
    """
    Extrae el valor Acc=XX.XX del archivo HResults.
    """
    with open(file_path, "r") as f:
        content = f.read()

    match = re.search(r'Acc=([0-9]+\.[0-9]+)', content)

    if match:
        return float(match.group(1))
    else:
        raise ValueError(f"Accuracy not found for {file_path}")

In [4]:
for STATES in STATES_LIST:

    RESULTS_ROOT = os.path.join(
        RESULTS_DIR,
        f"results_{STATES}_states"
    )

    # Nombre del excel
    OUTPUT_EXCEL = os.path.join(
        RESULTS_ROOT,
        f"HMM_Results_{STATES}_states.xlsx"
    )

    print(f"\nGenerando {OUTPUT_EXCEL}")

    with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:

        for OUTER in range(1, 11):

            outer_str = f"{OUTER:02d}"

            # Tabla vacía
            df = pd.DataFrame(
                index=[f"Group{outer_str}_{i:02d}" for i in range(1, 11)],
                columns=[f"{g}_gaussians" for g in range(1, 11)],
                dtype=float
            )

            for INNER in range(1, 11):

                inner_str = f"{INNER:02d}"

                for GAUSSIAN in range(1, 11):

                    results_file = os.path.join(
                        RESULTS_ROOT,
                        f"Group_{outer_str}",
                        f"Group{outer_str}_{inner_str}",
                        f"{GAUSSIAN}_gaussians",
                        "HResults",
                        f"results{outer_str}_state{inner_str}.txt"
                    )

                    if os.path.exists(results_file):

                        acc = extract_accuracy(results_file)

                        row_name = f"Group{outer_str}_{inner_str}"
                        col_name = f"{GAUSSIAN}_gaussians"

                        df.loc[row_name, col_name] = acc

                    else:
                        print(f"No encontrado: {results_file}")


            # Media por fila
            df["mean"] = df.mean(axis=1)

            # Media por columna
            mean_row = df.mean(axis=0)

            # Añadir fila mean
            df.loc["mean"] = mean_row


            sheet_name = f"Group_{outer_str}"

            df.to_excel(writer, sheet_name=sheet_name)

    print(f"Excel generado: {OUTPUT_EXCEL}")

print("All Excel files were generated correctly.")


Generando ./results\results_3_states\HMM_Results_3_states.xlsx
Excel generado: ./results\results_3_states\HMM_Results_3_states.xlsx

Generando ./results\results_4_states\HMM_Results_4_states.xlsx
Excel generado: ./results\results_4_states\HMM_Results_4_states.xlsx

Generando ./results\results_5_states\HMM_Results_5_states.xlsx
Excel generado: ./results\results_5_states\HMM_Results_5_states.xlsx
All Excel files were generated correctly.
